# Detector de Caracteres — Treino (YOLO)

Notebook principal de treino do detector agnóstico de caracteres (classe única `glifo`), etapa 1 do pipeline detector → classificador N1 → filtro N1.

Dataset de anotações: `miguelmussalam/manga109-character-bouding-box` (Kaggle) — páginas do Manga109 com bboxes em nível de caractere, anotadas via Roboflow. Também usa o Manga109 completo (imagens + anotações `<text>`) pra gerar dado sintético de treino (ver seção 4.5).

**Checklist antes de rodar:**
1. Anexe o dataset de anotações (Roboflow) e o dataset do Manga109 completo no painel lateral direito em **+ Add Input → Datasets**.
2. Habilite a GPU: *Session options → Accelerator → GPU T4 x2 ou P100*.
3. **Run All**.

In [ ]:
import subprocess
result = subprocess.run(["nvidia-smi"], capture_output=True, text=True)
print(result.stdout if result.returncode == 0 else "GPU nao encontrada!")

## 1. Parâmetros do experimento

**Altere apenas esta célula** para controlar o treino inteiro. As variáveis `KD_*` são lidas por `config.py` via `os.environ` — sem elas definidas, `config.py` usa seus próprios defaults.

In [ ]:
import os

# --- Modelo e treino YOLO ---
YOLO_MODEL      = "yolo26n.pt"   # ou "yolo11n.pt", "yolo11s.pt", etc.
EPOCHS          = 150
IMGSZ           = 1024           # caracteres pequenos: 1024 preserva mais detalhe que 640
BATCH           = 8              # ajustar para caber na VRAM (T4/P100 = 16GB); OOM -> reduzir

# --- Workers do dataloader (paralelismo de I/O) ---
KAGGLE_WORKERS  = 2              # T4/P100 do Kaggle tem 2 vCPUs
LOCAL_WORKERS   = 4

# --- Organizacao dos runs (pasta de resultados) ---
PROJECT_NAME    = "kanji_detector"

# ============================================================
# Aplica as configuracoes como variaveis de ambiente
# (KD_DATA_YAML e setado automaticamente na Celula 10, depois do split)
# ============================================================
os.environ["KD_YOLO_MODEL"]     = str(YOLO_MODEL)
os.environ["KD_EPOCHS"]         = str(EPOCHS)
os.environ["KD_IMGSZ"]          = str(IMGSZ)
os.environ["KD_BATCH"]          = str(BATCH)
os.environ["KD_KAGGLE_WORKERS"] = str(KAGGLE_WORKERS)
os.environ["KD_LOCAL_WORKERS"]  = str(LOCAL_WORKERS)
os.environ["KD_PROJECT_NAME"]   = str(PROJECT_NAME)

print("Parametros registrados!")
print(f"  Modelo: {YOLO_MODEL} | Epochs: {EPOCHS} | Imgsz: {IMGSZ} | Batch: {BATCH}")

## 2. Instalar dependências

In [ ]:
!pip install -q ultralytics scipy tqdm
print("Dependencias instaladas.")

## 3. Configurar repositório

Clona (ou atualiza, se já clonado nesta sessão) o repositório com o código do pipeline (`config.py`, `src/detector/train.py`) em `/kaggle/working/` e adiciona ao `sys.path` para permitir `from src.detector...`.

In [ ]:
import os
import sys
import shutil

WORK_DIR  = "/kaggle/working"
REPO_NAME = "Detector-de-kanjis-n1"
REPO_DIR  = os.path.join(WORK_DIR, REPO_NAME)
REPO_URL  = f"https://github.com/MiguelMussalam/{REPO_NAME}.git"

is_valid_repo = os.path.isdir(os.path.join(REPO_DIR, ".git"))

if not is_valid_repo:
    if os.path.exists(REPO_DIR):
        print(f"Diretorio {REPO_DIR} existe mas nao e um repo git valido. Removendo...")
        shutil.rmtree(REPO_DIR)
    print(f"Clonando {REPO_URL} ...")
    !git clone {REPO_URL} {REPO_DIR}
else:
    print("Repo valido encontrado. Atualizando...")
    !git -C {REPO_DIR} pull

assert os.path.isfile(os.path.join(REPO_DIR, "config.py")), \
    f"config.py nao encontrado em {REPO_DIR} — verifique se o clone funcionou"

os.chdir(REPO_DIR)
if REPO_DIR not in sys.path:
    sys.path.insert(0, REPO_DIR)

from config import PROJECT_NAME

print(f"Diretorio de trabalho: {os.getcwd()}")
print(f"Project name (runs):   {PROJECT_NAME}")

## 4. Preparar dataset

O dataset anotado (Kaggle Input) é read-only. Este passo localiza a pasta com `images/` e `labels/` dentro de `/kaggle/input/` (a estrutura de nesting varia conforme o ambiente Kaggle), copia para `/kaggle/working/` e cria um split determinístico train/val (85/15, seed fixa).

In [ ]:
import random
from pathlib import Path

def encontrar_dataset(root="/kaggle/input"):
    """
    Procura primeiro por um dataset que ja vem com split feito no Roboflow
    (pastas train/ e valid/, cada uma com images/ e labels/) -- nesse caso
    usamos o split EXATAMENTE como veio, sem misturar e redividir aqui.
    Se nao achar isso, cai para o modo antigo: uma unica pasta images/+labels/
    sem split, que a gente divide neste notebook.
    """
    for dirpath, dirnames, _ in os.walk(root):
        dirpath = Path(dirpath)
        tem_train = (dirpath / "train" / "images").is_dir() and (dirpath / "train" / "labels").is_dir()
        tem_valid = (dirpath / "valid" / "images").is_dir() and (dirpath / "valid" / "labels").is_dir()
        tem_test = (dirpath / "test" / "images").is_dir() and (dirpath / "test" / "labels").is_dir()
        if tem_train and (tem_valid or tem_test):
            nome_val = "valid" if tem_valid else "test"  # Roboflow pode chamar o 2o split de "valid" ou "test"
            return dirpath, nome_val

    for dirpath, dirnames, _ in os.walk(root):
        if "images" in dirnames and "labels" in dirnames:
            return Path(dirpath), False  # pasta unica, sem split

    return None, None

SRC, nome_split_val = encontrar_dataset()
split_pronto = bool(nome_split_val)

if SRC is None:
    raise FileNotFoundError(
        "Nenhum dataset valido encontrado em /kaggle/input. "
        "Verifique se o dataset foi anexado ao notebook."
    )

WORK = Path("/kaggle/working/manga_char")

if split_pronto:
    # ------------------------------------------------------------------
    # Split ja veio pronto do Roboflow (train/valid escolhidos por voce
    # na propria interface) -- so copia como esta, sem redividir.
    # ------------------------------------------------------------------
    print(f"Split ja feito no Roboflow encontrado em: {SRC}")
    for split_origem, split_destino in [("train", "train"), (nome_split_val, "val")]:
        src_img = SRC / split_origem / "images"
        src_lbl = SRC / split_origem / "labels"
        dst_img = WORK / split_destino / "images"
        dst_lbl = WORK / split_destino / "labels"
        dst_img.mkdir(parents=True, exist_ok=True)
        dst_lbl.mkdir(parents=True, exist_ok=True)
        for img_path in src_img.iterdir():
            if img_path.suffix.lower() not in {".jpg", ".jpeg", ".png"}:
                continue
            shutil.copy(img_path, dst_img / img_path.name)
            lbl_path = src_lbl / (img_path.stem + ".txt")
            if lbl_path.exists():
                shutil.copy(lbl_path, dst_lbl / lbl_path.name)
            else:
                print(f"AVISO: label ausente para {img_path.name}")

else:
    # ------------------------------------------------------------------
    # Pasta unica sem split (formato antigo) -- divide aqui no notebook.
    # Com poucas imagens, um split aleatorio pode por azar concentrar a
    # variedade toda de um lado. Prefira fazer o split direto no Roboflow
    # (train/valid na propria interface) em vez de depender deste modo.
    # ------------------------------------------------------------------
    print(f"Pasta unica (sem split previo) encontrada em: {SRC}")
    SRC_IMG = SRC / "images"
    SRC_LBL = SRC / "labels"

    imgs = sorted([p for p in SRC_IMG.iterdir() if p.suffix.lower() in {".jpg", ".jpeg", ".png"}])
    print(f"Total de imagens: {len(imgs)}")

    random.seed(42)
    imgs_embaralhadas = imgs.copy()
    random.shuffle(imgs_embaralhadas)
    val_count = max(1, round(len(imgs_embaralhadas) * 0.15))
    val_imgs = set(imgs_embaralhadas[:val_count])
    train_imgs = set(imgs_embaralhadas[val_count:])
    print("Usando split ALEATORIO (85/15, seed=42) -- considere fazer o split no Roboflow.")

    for split, group in [("train", train_imgs), ("val", val_imgs)]:
        (WORK / split / "images").mkdir(parents=True, exist_ok=True)
        (WORK / split / "labels").mkdir(parents=True, exist_ok=True)
        for img_path in group:
            lbl_path = SRC_LBL / (img_path.stem + ".txt")
            shutil.copy(img_path, WORK / split / "images" / img_path.name)
            if lbl_path.exists():
                shutil.copy(lbl_path, WORK / split / "labels" / lbl_path.name)
            else:
                print(f"AVISO: label ausente para {img_path.name}")

print(f"\nTreino: {len(list((WORK/'train/images').iterdir()))} imgs")
print(f"Val:    {len(list((WORK/'val/images').iterdir()))} imgs")


## 4.5 Gerar dataset sintético e fundir no TRAIN

Compõe glifos sintéticos ancorados em bboxes de linha `<text>` oficiais do Manga109 (ver `src/detector/synth_page.py`) -- resolve a escassez do dataset real anotado via Roboflow (~17 imagens). Só entra no split de **treino**; `val` continua 100% real, pra manter a avaliação (mAP) limpa. Sempre gera um contact sheet (`DETSYN_OUTPUT_DIR/contact_sheet.jpg`) -- audite visualmente antes de confiar no treino (principal risco: qualidade do inpaint em fundo texturizado/blocos de texto densos multi-linha).

In [ ]:
from config import FONTES_URL, FONTS_DIR
from src.helper.fonts import download_fonts

print("[INFO] Baixando fontes...")
download_fonts(FONTES_URL, FONTS_DIR)

In [ ]:
N_PAGINAS_SINTETICAS = 300  # ajuste conforme necessario

!python -m src.detector.generate_pages --n-pages {N_PAGINAS_SINTETICAS}


In [ ]:
import shutil
from pathlib import Path
from IPython.display import Image as IPImage, display

from config import DETSYN_OUTPUT_DIR

synth_img_dir = Path(DETSYN_OUTPUT_DIR) / "images"
synth_lbl_dir = Path(DETSYN_OUTPUT_DIR) / "labels"

dst_img = WORK / "train" / "images"
dst_lbl = WORK / "train" / "labels"

n_copiadas = 0
for img_path in synth_img_dir.glob("*.jpg"):
    lbl_path = synth_lbl_dir / (img_path.stem + ".txt")
    if not lbl_path.exists():
        continue
    shutil.copy(img_path, dst_img / img_path.name)
    shutil.copy(lbl_path, dst_lbl / lbl_path.name)
    n_copiadas += 1

print(f"{n_copiadas} paginas sinteticas fundidas em {dst_img}")
print(f"Treino total agora: {len(list(dst_img.iterdir()))} imgs "
      f"(real + sintetico -- val em {WORK / 'val' / 'images'} continua so real)")

contact_sheet = Path(DETSYN_OUTPUT_DIR) / "contact_sheet.jpg"
if contact_sheet.exists():
    print("\nContact sheet (auditoria visual -- confira antes de treinar):")
    display(IPImage(str(contact_sheet)))


## 5. Gerar `data.yaml`

In [ ]:
import yaml

data_yaml = {
    "path": str(WORK),
    "train": "train/images",
    "val": "val/images",
    "nc": 1,
    "names": ["glifo"],
}

yaml_path = WORK / "data.yaml"
with open(yaml_path, "w") as f:
    yaml.safe_dump(data_yaml, f, sort_keys=False)

os.environ["KD_DATA_YAML"] = str(yaml_path)

print(yaml_path.read_text())
print(f"KD_DATA_YAML setado para: {yaml_path}")

## 6. Treinar

Chama `src.detector.train`, que lê `DATA_YAML` e os hiperparâmetros (`YOLO_MODEL`, `EPOCHS`, `IMGSZ`, `BATCH`, workers) do `config.py`. Para ajustar algum hiperparâmetro nesta sessão, defina a env var correspondente (`KD_EPOCHS`, `KD_BATCH`, etc.) antes de rodar a célula abaixo.

In [ ]:
!python -m src.detector.train

## 7. Resultados e curvas

In [ ]:
import pandas as pd
from IPython.display import Image, display

# O train.py grava o caminho real de saida nesse arquivo -- mais confiavel
# do que reconstruir o caminho aqui (versoes do Ultralytics podem mudar
# como resolvem o "project", ex: prefixar "runs/<task>/").
marcador_path = os.path.join(REPO_DIR, "ultimo_run_detector.txt")
if os.path.exists(marcador_path):
    with open(marcador_path, encoding="utf-8") as f:
        RUNS_DIR = Path(f.read().strip())
else:
    print("[AVISO] Arquivo marcador nao encontrado, tentando caminho padrao (pode estar errado).")
    RUNS_DIR = Path(REPO_DIR) / PROJECT_NAME / "run"

print(f"RUNS_DIR: {RUNS_DIR}")
results_csv = RUNS_DIR / "results.csv"

if results_csv.exists():
    df = pd.read_csv(results_csv)
    df.columns = [c.strip() for c in df.columns]
    last = df.iloc[-1]
    print(f"mAP@50:      {last.get('metrics/mAP50(B)', float('nan')):.4f}")
    print(f"mAP@50-95:   {last.get('metrics/mAP50-95(B)', float('nan')):.4f}")
    print(f"Precision:   {last.get('metrics/precision(B)', float('nan')):.4f}")
    print(f"Recall:      {last.get('metrics/recall(B)', float('nan')):.4f}")
else:
    print(f"results.csv nao encontrado em {results_csv}")

for fname, titulo in [
    ("results.png",          "=== Curvas de Treino (mAP / Loss) ==="),
    ("confusion_matrix.png", "=== Matriz de Confusao ==="),
    ("val_batch0_pred.jpg",  "=== Predicoes no conjunto de validacao ==="),
]:
    fpath = RUNS_DIR / fname
    if fpath.exists():
        print(titulo)
        display(Image(str(fpath)))


## 8. Inspeção visual em página fora do treino

Roda o `best.pt` numa página de manga não usada no treino/validação — ajuste `pagina_teste` abaixo. Parâmetros de inferência calibrados na rodada anterior: `conf=0.30`, `iou=0.40`, `max_det=1000` (páginas de manga têm 200+ caracteres, acima do default de 300 detecções).

In [ ]:
from ultralytics import YOLO

best = YOLO(str(RUNS_DIR / "weights" / "best.pt"))

# Ajuste para o caminho de uma pagina de manga fora do dataset de treino/validacao
pagina_teste = "/kaggle/input/CAMINHO/PARA/pagina_teste.jpg"

if os.path.exists(pagina_teste):
    results = best.predict(
        pagina_teste,
        imgsz=1024,
        conf=0.30,       # sobe um pouco para reduzir bboxes marginais
        iou=0.40,        # NMS mais agressivo para eliminar duplicatas
        agnostic_nms=True,
        max_det=1000,    # default e 300, paginas tem 200+ caracteres
        save=True,
        project="/kaggle/working/predict",
        name="fora_do_treino",
    )
    for r in results:
        print(f"{Path(r.path).name}: {len(r.boxes)} deteccoes")
else:
    print(f"Ajuste 'pagina_teste' para uma imagem valida. Caminho atual nao existe: {pagina_teste}")

## 9. Compactar e baixar

In [ ]:
import zipfile
from IPython.display import FileLink, display

zip_name = "/kaggle/working/resultados_detector.zip"

def zipdir(path, ziph, arcbase):
    for root, dirs, files in os.walk(path):
        for file in files:
            filepath = os.path.join(root, file)
            arcname  = os.path.join(arcbase, os.path.relpath(filepath, path))
            ziph.write(filepath, arcname)

print(f"Criando {zip_name}...")
with zipfile.ZipFile(zip_name, "w", zipfile.ZIP_DEFLATED) as zipf:
    if RUNS_DIR.exists():
        zipdir(str(RUNS_DIR), zipf, "run")
        print("Resultados YOLO adicionados.")
    else:
        print(f"AVISO: pasta de runs nao encontrada: {RUNS_DIR}")

    config_file = os.path.join(REPO_DIR, "config.py")
    if os.path.exists(config_file):
        zipf.write(config_file, "config.py")
        print("config.py adicionado.")

print(f"\nZip criado: {zip_name}")
display(FileLink(zip_name))